# Treinamento v3.1 (SFT Puro, Zero Vazamento de Inglês) — Especialista Acadêmico `luiz-academico`
**O que mudou na v3.1:**
1. **Corpus Completo (415 exemplos únicos 100% Português Humano):** Junta todos os **142 pares Ouro do Luiz** (86%–99% Humano no GPTZero) com **todos os 273 blocos da dissertação do Dr. Marcus Braga** (100% Humano no GPTZero).
2. **SFT Puro (SEM DPO):** Removemos a etapa de DPO porque ela empurrava tokens do Llama 3.1 para o inglês. Com 415 exemplos SFT + `ResponseOnlyCollator`, o modelo mantém 100% de fluência em português e paridade estrita de parágrafos.
3. **Exportação Blindada:** Faz a conversão temporária no disco local do Colab (`/content/tmp_gguf`) com limite baixo de RAM (`0.35`) e copia apenas o arquivo final de 4.9 GB (`luiz-academico-v3_1-q4_k_m.gguf`) para o seu Google Drive.

In [ ]:
# CÉLULA 1: Conectar Google Drive e garantir seed_v3_1_corpus.jsonl
import os, shutil
from google.colab import drive, files

drive.mount('/content/drive')

drive_seed = "/content/drive/MyDrive/seed_v3_1_corpus.jsonl"
local_seed = "seed_v3_1_corpus.jsonl"

if os.path.exists(local_seed):
    shutil.copy(local_seed, drive_seed)
elif not os.path.exists(drive_seed):
    print("Selecione o arquivo 'seed_v3_1_corpus.jsonl' do seu computador:")
    uploaded = files.upload()
    up_name = list(uploaded.keys())[0]
    shutil.copy(up_name, drive_seed)

shutil.copy(drive_seed, local_seed)
print(f"✅ Google Drive conectado e {local_seed} pronto!")

In [ ]:
# CÉLULA 2: Instalar Unsloth e dependências
%%capture
!pip install unsloth
!pip uninstall unsloth -y && pip install --upgrade --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git

In [ ]:
# CÉLULA 3: Carregar Llama 3.1 8B Base e gerar entradas 'IA-ificadas difíceis' para os 273 blocos
import json, os, shutil, torch
from unsloth import FastLanguageModel
from tqdm.auto import tqdm

max_seq_length = 2048
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = True,
)

drive_v3_1 = "/content/drive/MyDrive/dataset_treino_v3_1.jsonl"
local_v3_1 = "dataset_treino_v3_1.jsonl"

if os.path.exists(drive_v3_1):
    shutil.copy(drive_v3_1, local_v3_1)
    print(f"✅ Dataset v3.1 já encontrado no Drive ({drive_v3_1}), pulando geração!")
else:
    FastLanguageModel.for_inference(model)
    tokenizer.padding_side = "left"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    rows = [json.loads(line) for line in open("seed_v3_1_corpus.jsonl", encoding="utf-8")]
    to_gen = [r for r in rows if r.get("needs_ai_gen")]
    ready = [r for r in rows if not r.get("needs_ai_gen")]

    print(f"⏳ Gerando versões IA-ificadas em português para {len(to_gen)} blocos humanos na GPU T4...")
    sys_msg = "Você é um assistente acadêmico padrão. Reescreva o texto estritamente em português brasileiro formal usando frases diretas (Sujeito-Verbo-Objeto) de tamanho médio e conectivos comuns (como 'Nesse sentido', 'Além disso', 'Vale ressaltar', 'Portanto'), mantendo rigorosamente todos os fatos e o mesmo número de parágrafos, sem comentários extras."

    batch_size = 6
    for i in tqdm(range(0, len(to_gen), batch_size)):
        batch = to_gen[i:i+batch_size]
        prompts = []
        for item in batch:
            msgs = [
                {"role": "system", "content": sys_msg},
                {"role": "user", "content": f"Reescreva o texto abaixo em português brasileiro:\n\n{item['output']}"}
            ]
            prompts.append(tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True))

        inputs = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True, max_length=1024).to("cuda")
        with torch.no_grad():
            outputs = model.generate(**inputs, max_new_tokens=500, temperature=0.7, top_p=0.9, do_sample=True, pad_token_id=tokenizer.pad_token_id)
        
        in_len = inputs["input_ids"].shape[1]
        for item, out_ids in zip(batch, outputs):
            gen_txt = tokenizer.decode(out_ids[in_len:], skip_special_tokens=True).strip()
            out_paras = len([p for p in item["output"].split("\n\n") if p.strip()])
            gen_paras = [p.strip() for p in gen_txt.split("\n\n") if p.strip()]
            if len(gen_paras) != out_paras and out_paras == 1:
                gen_txt = " ".join(gen_paras)
            item["input"] = gen_txt
            item["needs_ai_gen"] = False

    all_v3_1 = ready + to_gen
    with open(local_v3_1, "w", encoding="utf-8") as f:
        for r in all_v3_1:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    shutil.copy(local_v3_1, drive_v3_1)
    print(f"✅ {len(all_v3_1)} exemplos salvos em {drive_v3_1}!")

In [ ]:
# CÉLULA 4: Treino SFT Puro (415 exemplos, QLoRA r=32, alpha=64, ResponseOnlyCollator)
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments, DataCollatorForLanguageModeling
from unsloth import FastLanguageModel, is_bfloat16_supported
import gc, torch

model = FastLanguageModel.get_peft_model(
    model,
    r = 32,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 64,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

alpaca_prompt = """Abaixo está uma instrução que descreve uma tarefa, acompanhada de uma entrada que fornece o contexto. Escreva uma resposta que conclua adequadamente a solicitação sem qualquer metatexto, mantendo exatamente o mesmo número de parágrafos e todos os fatos da entrada.

### Instrução:
{}

### Entrada:
{}

### Resposta:
{}"""

EOS_TOKEN = tokenizer.eos_token
def formatting_prompts_func(examples):
    texts = [
        alpaca_prompt.format(inst, inp, out) + EOS_TOKEN
        for inst, inp, out in zip(examples["instruction"], examples["input"], examples["output"])
    ]
    return {"text": texts}

sft_dataset = load_dataset("json", data_files="dataset_treino_v3_1.jsonl", split="train")
sft_dataset = sft_dataset.map(formatting_prompts_func, batched=True)

class ResponseOnlyCollator(DataCollatorForLanguageModeling):
    def __init__(self, tokenizer):
        super().__init__(tokenizer=tokenizer, mlm=False)
        self.resp_ids = tokenizer.encode("### Resposta:\n", add_special_tokens=False)
    def torch_call(self, examples):
        batch = super().torch_call(examples)
        labels = batch["labels"]
        m = len(self.resp_ids)
        for i in range(labels.size(0)):
            seq = labels[i].tolist()
            for j in range(len(seq) - m + 1):
                if seq[j:j+m] == self.resp_ids:
                    labels[i, :j+m] = -100
                    break
        return batch

pasta_v3_1 = "/content/drive/MyDrive/lora_luiz_v3_1_final"
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = sft_dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    data_collator = ResponseOnlyCollator(tokenizer),
    dataset_num_proc = 2,
    packing = False,
    neftune_noise_alpha = 5,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 12,
        num_train_epochs = 3,
        learning_rate = 1.5e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 10,
        save_strategy = "epoch",
        optim = "adamw_8bit",
        weight_decay = 0.03,
        lr_scheduler_type = "cosine",
        seed = 3407,
        output_dir = pasta_v3_1,
        report_to = "none",
    ),
)

trainer.train()
model.save_pretrained(pasta_v3_1)
tokenizer.save_pretrained(pasta_v3_1)
del trainer, sft_dataset
gc.collect(); torch.cuda.empty_cache()
print(f"✅ Treino v3.1 (SFT Puro) salvo definitivamente no Drive em: {pasta_v3_1}")

In [ ]:
# CÉLULA 5: Exportar GGUF no disco local do Colab (/content/tmp_gguf) e copiar só o .gguf final para o Drive
import glob, shutil, os, gc, torch

gc.collect(); torch.cuda.empty_cache()
!rm -rf /content/tmp_gguf*

model.save_pretrained_gguf(
    "/content/tmp_gguf",
    tokenizer,
    quantization_method = "q4_k_m",
    maximum_memory_usage = 0.35,
)

ggufs = glob.glob("/content/tmp_gguf*/*Q4_K_M.gguf")
if ggufs:
    dst = "/content/drive/MyDrive/luiz-academico-v3_1-q4_k_m.gguf"
    shutil.copy(ggufs[0], dst)
    print(f"🎉 SUCESSO TOTAL! Modelo v3.1 (4.9 GB) salvo no Drive em: {dst}")
else:
    print("⚠️ Arquivo não encontrado em /content/tmp_gguf*. Conteúdo:", glob.glob("/content/tmp_gguf*/*"))